In [1]:
!git clone https://github.com/Brayan695/RNAseq-AMD.git

Cloning into 'RNAseq-AMD'...
remote: Enumerating objects: 2738, done.
remote: Counting objects: 100% (758/758), done.
remote: Compressing objects: 100% (555/555), done.
remote: Total 2738 (delta 341), reused 576 (delta 195), pack-reused 1980 (from 1)
Receiving objects: 100% (2738/2738), 1.05 GiB | 26.00 MiB/s, done.
Resolving deltas: 100% (1000/1000), done.
Updating files: 100% (1876/1876), done.


In [2]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_selection import f_classif
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [3]:
DATA_PATH = "/kaggle/working/RNAseq-AMD/Dataset/MetaSheet_Processed.csv"
N_ITERATIONS = 1000
SAMPLE_FRACTION = 0.8
TOP_K_FRACTION = 0.30     #per iteration: keep top 30% of usable features by score
TOP_N_FRACTION = 0.10     #final: keep top 10% of usable features by selection frequency                                      
RANDOM_SEED = 2026

# All 6 pairwise MGS stage comparisons to run. Each tuple is (negative_class_stage, positive_class_stage) 
# for ex: (1, 4) means "control (MGS1) vs. late AMD (MGS4)"
STAGE_PAIRS = [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]

EXCLUDE_LABEL_LEAKAGE = True
LEAKAGE_COLUMNS = [
    "oc_AMD",
    "oc_dry AMD",
    "oc_macular degeneration",
    "oc_macular degeneraton",
    "oc_AMD (Avastin injections)",
    "oc_AMD (Lucentis injections)",
    "oc_AMD (OD)",
    "oc_AMD (took vitamins)",
    "oc_AMD (wet)",
    "oc_AMD -took vitamins",
    "oc_AMD?",
    "oc_Wet AMD",
    "oc_wet AMD",
    "oc_early AMD",
    "oc_possible AMD",
    "oc_possible macular degeneration",
    "oc_AMD (received shots)",
]

COLLAPSE_MAP_PATH = "/kaggle/working/RNAseq-AMD/Dataset/MetaSheet_Feature_Selection.csv"

EXCLUDE_CONFOUNDS = True
CONFOUND_COLUMNS = ["age"]

EXCLUDE_NO_INFO = True
# Placeholder/blank chart-note entries -- not a real clinical fact, just "nothing was
# recorded" or "unknown" written into the column name itself.
NO_INFO_COLUMNS = ["mh_-", "mh_N/A", "oc_?", "oc_n/a", "oc_unknown"]

rng = np.random.default_rng(RANDOM_SEED)

In [4]:
def load_data(path, stage_pair=(1, 4)):
    """
    Load metadata, filter to two MGS stages, split into features (X) and
    label (y). y = 1 for the higher numbered (more advanced) stage in
    stage_pair, 0 for the lower numbered stage. All other stages are
    excluded from this comparison entirely.
    """
    neg_stage, pos_stage = stage_pair
    df = pd.read_csv(path)
    df = df[df["mgs_level"].isin([neg_stage, pos_stage])].reset_index(drop=True)
    y = (df["mgs_level"] == pos_stage).astype(int).values
    drop_cols = [c for c in ["sample_id", "mgs_level"] if c in df.columns]
    X = df.drop(columns=drop_cols)
    return X, y

In [5]:
def drop_constant_features(X):
    """Remove features with zero variance"""
    nunique = X.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()
    X_filtered = X.drop(columns=constant_cols)
    return X_filtered, constant_cols

## Column review: full list after dropping leakage/confound/zero-variance columns

Run this once, on the full (all-stage) dataset, to get a single alphabetized column list to manually scan for near-duplicate features before consolidating and re-running feature selection.

In [6]:
# --- Full column list for manual review (near-duplicate consolidation) ---
# Loads ALL 4 MGS stages (not filtered to one pairwise comparison), applies
# the same label-leakage and confound drops as the main pipeline, then drops
# zero-variance columns. This gives ONE comprehensive, alphabetically-sorted
# column list to scan for near-duplicate features (e.g. "mh_high chol" vs
# "mh_high cholesterol") before re-running feature selection. Alphabetical
# sort naturally groups similarly-named columns next to each other.

df_full = pd.read_csv(DATA_PATH)
drop_cols = [c for c in ["sample_id", "mgs_level"] if c in df_full.columns]
X_full = df_full.drop(columns=drop_cols)

if EXCLUDE_LABEL_LEAKAGE:
    present = [c for c in LEAKAGE_COLUMNS if c in X_full.columns]
    X_full = X_full.drop(columns=present)
    print(f"dropped {len(present)} label-leakage columns")

if EXCLUDE_CONFOUNDS:
    present = [c for c in CONFOUND_COLUMNS if c in X_full.columns]
    X_full = X_full.drop(columns=present)
    print(f"dropped {len(present)} confound columns")

if EXCLUDE_NO_INFO:
    present = [c for c in NO_INFO_COLUMNS if c in X_full.columns]
    X_full = X_full.drop(columns=present)
    print(f"dropped {len(present)} no-info/placeholder columns: {present}")

X_full_nonconstant, dropped_full = drop_constant_features(X_full)

print(f"\nfull dataset (all 4 stages): {X_full.shape[1]} columns after leakage/confound drop")
print(f"dropped {len(dropped_full)} constant (zero-variance) columns")
print(f"{X_full_nonconstant.shape[1]} usable columns remain\n")

remaining_cols = sorted(X_full_nonconstant.columns.tolist())
for c in remaining_cols:
    print(c)

pd.Series(remaining_cols, name="column").to_csv(
    "metadata_columns_for_review.csv", index=False
)
print(f"\nSaved: metadata_columns_for_review.csv ({len(remaining_cols)} columns)")

dropped 17 label-leakage columns
dropped 1 confound columns
dropped 5 no-info/placeholder columns: ['mh_-', 'mh_N/A', 'oc_?', 'oc_n/a', 'oc_unknown']

full dataset (all 4 stages): 591 columns after leakage/confound drop
dropped 0 constant (zero-variance) columns
591 usable columns remain

A69S_GG
A69S_GT
A69S_TT
Y402H_CC
Y402H_CT
Y402H_TT
mh_1/2 pack day for 60 yrs; HBP
mh_A Fib
mh_A fib
mh_A. Fib
mh_A. fib
mh_A. fib (pacemaker) - 2007; high cholesterol)
mh_AA repair
mh_AAA
mh_AML
mh_ARDS
mh_Acute Renal Failure
mh_Afib
mh_Alcohol abuse
mh_Alcoholic Cirrhosis
mh_Alzheimer's
mh_Alzheimer's depression
mh_Alzheimers
mh_Anemia
mh_Aneurysm
mh_Bladder cancer with mets
mh_Breast cancer
mh_C.diff
mh_CABG
mh_CABG - 1979
mh_CAD
mh_CAD (CABG - 1998)
mh_CAD s/p CABG
mh_CAD s/p stent
mh_CHF
mh_CKD
mh_CKD - stage 3
mh_CLL
mh_CMF+Afib post op
mh_CML
mh_COPD
mh_CVA
mh_Cardiomyopathy
mh_Cerebrovascular accident
mh_Chronic Kidney disease
mh_Coronary Artery Disease
mh_DJD
mh_Degenerative Joint disease
mh_

## Collapse near-duplicate columns

Loads the annotated `Feature: / Flag: / Collapse Group: / Notes:` CSV produced from the manual
review above. Any row with a non-empty `Collapse Group:` gets merged into one column with its
siblings; every source column in a group should represent the identical clinical fact for a
given patient, so the merge rule is **logical OR** (`max` across 0/1 columns) if any of the
duplicate-spelling columns is 1 for a patient, the collapsed column is 1.

Rows with an empty `Collapse Group:` (including anything flagged as label leakage) pass through
unchanged. they either don't have a literal duplicate, or they're excluded upstream by
`LEAKAGE_COLUMNS` already.

In [7]:
def load_collapse_mapping(path):
    """Build a raw-column -> canonical-name mapping from the annotated review CSV.
    Rows with a non-empty 'Collapse Group:' map to that group name (their merge
    target). Rows with an empty Collapse Group map to themselves (no merge)."""
    review = pd.read_csv(path)
    mapping = {}
    for _, row in review.iterrows():
        feat = row["Feature:"]
        collapse = row.get("Collapse Group:", np.nan)
        if pd.isna(collapse) or str(collapse).strip() == "":
            mapping[feat] = feat
        else:
            mapping[feat] = str(collapse).strip()
    return mapping

def collapse_duplicate_columns(X, mapping, verbose=True):
    """Merge near-duplicate columns in X per `mapping` (raw col -> canonical name).
    Columns that map to the same canonical name are combined via logical OR
    (elementwise max on the 0/1 columns). Columns not present in `mapping` at
    all (e.g. genotype/sex_bin, or anything not in the review CSV) pass through
    untouched under their original name."""
    canonical_to_raw = {}
    for col in X.columns:
        canonical = mapping.get(col, col)
        canonical_to_raw.setdefault(canonical, []).append(col)

    merged_data = {}
    n_merged_groups = 0
    n_cols_absorbed = 0
    for canonical, raw_cols in canonical_to_raw.items():
        if len(raw_cols) == 1:
            merged_data[canonical] = X[raw_cols[0]]
        else:
            merged_data[canonical] = X[raw_cols].max(axis=1)
            n_merged_groups += 1
            n_cols_absorbed += len(raw_cols) - 1

    X_collapsed = pd.DataFrame(merged_data, index=X.index)
    if verbose:
        print(f"  collapsed {n_merged_groups} groups, absorbing {n_cols_absorbed} "
              f"duplicate columns ({X.shape[1]} -> {X_collapsed.shape[1]} columns)")
    return X_collapsed


COLLAPSE_MAPPING = load_collapse_mapping(COLLAPSE_MAP_PATH)
print(f"Loaded collapse mapping for {len(COLLAPSE_MAPPING)} reviewed columns "
      f"({len(set(COLLAPSE_MAPPING.values()))} distinct canonical targets)")


Loaded collapse mapping for 604 reviewed columns (259 distinct canonical targets)


In [8]:
def anova_scores(X, y):
    """ANOVA F-test score per feature, all features at once."""
    f_stat, _ = f_classif(X.values, y)
    f_stat = np.nan_to_num(f_stat, nan=0.0)
    return pd.Series(f_stat, index=X.columns)
 
def auc_scores(X, y):
    """AUC per feature, computed directly from ranks (equivalent to
    roc_auc_score but vectorized across every column at once)
    Take max(auc, 1-auc) so the direction of the association (e.g.
    presence vs. absence of a condition) doesn't penalize the score."""
    n1 = y.sum()
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return pd.Series(0.5, index=X.columns)
    ranks = stats.rankdata(X.values, axis=0, method="average")
    sum_ranks_pos = ranks[y == 1].sum(axis=0)
    auc = (sum_ranks_pos - n1 * (n1 + 1) / 2) / (n1 * n0)
    auc = np.maximum(auc, 1 - auc)
    return pd.Series(auc, index=X.columns)
 
def kruskal_scores(X, y):
    """Kruskal Wallis H statistic per feature  computed directly from ranks"""
    N = len(y)
    ranks = stats.rankdata(X.values, axis=0, method="average")
    n1 = y.sum()
    n0 = N - n1
    R1 = ranks[y == 1].sum(axis=0)
    R0 = ranks[y == 0].sum(axis=0)
    H = (12 / (N * (N + 1))) * ((R1 ** 2) / n1 + (R0 ** 2) / n0) - 3 * (N + 1)
    # Tie correction: C = 1 - sum(t^3 - t) / (N^3 - N) computed per column
    tie_correction = np.ones(X.shape[1])
    X_vals = X.values
    for i in range(X.shape[1]):
        _, counts = np.unique(X_vals[:, i], return_counts=True)
        tie_sum = np.sum(counts ** 3 - counts)
        tie_correction[i] = 1 - tie_sum / (N ** 3 - N)
    with np.errstate(divide="ignore", invalid="ignore"):
        H_corrected = np.where(tie_correction > 0, H / tie_correction, 0.0)
    H_corrected = np.nan_to_num(H_corrected, nan=0.0, posinf=0.0, neginf=0.0)
    return pd.Series(H_corrected, index=X.columns)

In [9]:
def run_pipeline(X, y, n_iterations, sample_fraction, top_k, seed):
    """Run the 1000 iteration resampling + scoring loop.
    Returns a DataFrame of selection frequency (0-1) per feature per method.
    """
    counts = {
        "anova": pd.Series(0, index=X.columns, dtype=int),
        "auc": pd.Series(0, index=X.columns, dtype=int),
        "kruskal": pd.Series(0, index=X.columns, dtype=int),
    }
    for it in range(n_iterations):
        #Stratified 80% resample: preserves the control to AMD ratio in
        # every iteration so small class features aren't starved.
        X_sub, _, y_sub, _ = train_test_split(
            X, y,
            train_size=sample_fraction,
            stratify=y,
            random_state=RANDOM_SEED + it,
        )
        scores = {
            "anova": anova_scores(X_sub, y_sub),
            "auc": auc_scores(X_sub, y_sub),
            "kruskal": kruskal_scores(X_sub, y_sub),
        }
        for method, s in scores.items():
            top_features = s.sort_values(ascending=False).head(top_k).index
            counts[method].loc[top_features] += 1
        if (it + 1) % 100 == 0:
            print(f"  iteration {it + 1}/{n_iterations} done")
    freq = pd.DataFrame({m: c / n_iterations for m, c in counts.items()})
    return freq

In [10]:
def select_consistent_features(freq_df, top_n):
    """Take the top N features per method (by selection frequency), then
    intersect across all three methods."""
    top_sets = {}
    for method in freq_df.columns:
        top_sets[method] = set(
            freq_df[method].sort_values(ascending=False).head(top_n).index
        )
    consistent = top_sets["anova"] & top_sets["auc"] & top_sets["kruskal"]
    return consistent, top_sets

In [11]:
all_results = {}
 
for stage_pair in STAGE_PAIRS:
    neg_stage, pos_stage = stage_pair
    label = f"{neg_stage}v{pos_stage}"
    print(f"\n{'='*60}\nStage comparison: MGS{neg_stage} vs. MGS{pos_stage}\n{'='*60}")
 
    X_raw, y = load_data(DATA_PATH, stage_pair=stage_pair)
    print(f"  {X_raw.shape[0]} samples, {X_raw.shape[1]} raw features")
    print(f"  class balance: {sum(y==0)} MGS{neg_stage}, {sum(y==1)} MGS{pos_stage}")
 
    if EXCLUDE_LABEL_LEAKAGE:
        present = [c for c in LEAKAGE_COLUMNS if c in X_raw.columns]
        X_raw = X_raw.drop(columns=present)
        print(f"  dropped {len(present)} label-leakage columns (chart notes "
              f"that restate the AMD diagnosis): {present}")
 
    if EXCLUDE_CONFOUNDS:
        present = [c for c in CONFOUND_COLUMNS if c in X_raw.columns]
        X_raw = X_raw.drop(columns=present)
        print(f"  dropped {len(present)} confound columns: {present}")

    if EXCLUDE_NO_INFO:
        present = [c for c in NO_INFO_COLUMNS if c in X_raw.columns]
        X_raw = X_raw.drop(columns=present)
        print(f"  dropped {len(present)} no-info/placeholder columns: {present}")
 
    X_raw = collapse_duplicate_columns(X_raw, COLLAPSE_MAPPING)


    X, dropped = drop_constant_features(X_raw)
    print(f"  dropped {len(dropped)} constant (zero-variance) features")
 
    top_k = max(1, int(X.shape[1] * TOP_K_FRACTION))
    top_n = max(1, int(X.shape[1] * TOP_N_FRACTION))
    print(f"  per-iteration top-K = {top_k}, final top-N per method = {top_n}")
 
    freq_df = run_pipeline(X, y, N_ITERATIONS, SAMPLE_FRACTION, top_k, RANDOM_SEED)
    consistent, top_sets = select_consistent_features(freq_df, top_n)
 
    print(f"\n  Features selected by all 3 methods: {len(consistent)}")
    for f in sorted(consistent):
        print(f"    {f}  (anova={freq_df.loc[f,'anova']:.2f}, "
              f"auc={freq_df.loc[f,'auc']:.2f}, kruskal={freq_df.loc[f,'kruskal']:.2f})")
 
    freq_df.to_csv(f"metadata_feature_selection_frequencies_{label}.csv")
    pd.Series(sorted(consistent), name="feature").to_csv(
        f"metadata_selected_features_{label}.csv", index=False
    )
    print(f"  Saved: metadata_feature_selection_frequencies_{label}.csv, "
          f"metadata_selected_features_{label}.csv")
 
    all_results[label] = consistent


Stage comparison: MGS1 vs. MGS2
  280 samples, 614 raw features
  class balance: 105 MGS1, 175 MGS2
  dropped 17 label-leakage columns (chart notes that restate the AMD diagnosis): ['oc_AMD', 'oc_dry AMD', 'oc_macular degeneration', 'oc_macular degeneraton', 'oc_AMD (Avastin injections)', 'oc_AMD (Lucentis injections)', 'oc_AMD (OD)', 'oc_AMD (took vitamins)', 'oc_AMD (wet)', 'oc_AMD -took vitamins', 'oc_AMD?', 'oc_Wet AMD', 'oc_wet AMD', 'oc_early AMD', 'oc_possible AMD', 'oc_possible macular degeneration', 'oc_AMD (received shots)']
  dropped 1 confound columns: ['age']
  dropped 5 no-info/placeholder columns: ['mh_-', 'mh_N/A', 'oc_?', 'oc_n/a', 'oc_unknown']
  collapsed 63 groups, absorbing 345 duplicate columns (591 -> 246 columns)
  dropped 51 constant (zero-variance) features
  per-iteration top-K = 58, final top-N per method = 19
  iteration 100/1000 done
  iteration 200/1000 done
  iteration 300/1000 done
  iteration 400/1000 done
  iteration 500/1000 done
  iteration 600/100

In [12]:

#which features are robust across MULTIPLE stage comparisons not just one?

all_features = sorted(set().union(*all_results.values()))
summary = pd.DataFrame(index=all_features)
for label, feats in all_results.items():
    summary[label] = summary.index.isin(feats)
summary["n_pairs_selected"] = summary.sum(axis=1)
summary = summary.sort_values("n_pairs_selected", ascending=False)
 
print(f"\n{'='*60}\nCross-pair summary\n{'='*60}")
print(summary)
summary.to_csv("metadata_feature_selection_cross_pair_summary.csv")
print("\nSaved: metadata_feature_selection_cross_pair_summary.csv")


Cross-pair summary
                                                   1v2    1v3    1v4    2v3  \
Cataracts                                         True   True   True   True   
A69S_GG                                          False  False   True   True   
Pseudophakic                                     False   True   True   True   
Y402H_CC                                         False   True   True   True   
Alzheimer's disease                               True  False  False  False   
A69S_TT                                          False  False   True  False   
A69S_GT                                          False  False   True   True   
Y402H_TT                                         False  False   True   True   
mh_asthma                                         True   True   True  False   
Dementia (unspecified)                           False   True   True  False   
Coronary revascularization history (CABG/stent)  False   True  False   True   
sex_bin                         